<a href="https://colab.research.google.com/github/rohansangal1/Brain-Tumor-Classification/blob/main/Copy_of_Rohan_Brain_Tumor_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- Image removed: run notebook to regenerate -->

# **Brain Tumor Classification**

In the United States, there are over 200,000 cases of brain tumors yearly, putting it in the **Common** category, and worldwide, over 251,329 people died from them in 2020.

When Brain Tumors are left untreated, they grow more, which also leads to more aggressive effects such as seizures, dramatic changes in blood pressure, and extreme brain damage, eventually leading to death which can occur within a time span as low as 3-4 months.

In this project, Brain Tumor MRI images will be used to diganose and classify brain tumors. 

The importance of this is early detection and classification of brain tumors can help in selecting the best treatments to save the lives of many patients.

The dataset that will be used contains over 7000 images that are in one of four classes, which are Glioma, Notumor, Meningioma, and Pituitary.


*   Glioma - A tumor that forms when glial cells grow out of control.

*   Notumor - No tumor is present in the brain.

*   Meningioma - A usually noncancerous tumor that arises from the membranes surrounding the brain and spinal cord.

*   Pituitary - A tumor that forms in the pituitary gland near the brain that can cause changes in hormone levels in the body.

<!-- Image removed: run notebook to regenerate -->

In [ ]:
# Install dependencies (if running locally)
# pip install -r ../requirements.txt

import os
import math
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from skimage import color
from PIL import Image
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, confusion_matrix, roc_auc_score,
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
import keras
from keras.layers import Dense, Conv2D, BatchNormalization, Activation, MaxPooling2D, Dropout, Flatten
from keras.models import Sequential
from keras.applications.mobilenet import MobileNet

# Paths — update if running locally
TRAIN_PATH = "../data/Training"  # or "/content/Training" in Colab
TEST_PATH  = "../data/Testing"
labels = {"glioma": 0, "notumor": 1, "meningioma": 2, "pituitary": 3}


In [ ]:
# labels
labels = {'glioma':0,'notumor':1,'meningioma':2,'pituitary':3}

In [ ]:
labels

In [ ]:
!ls

In [ ]:
# os.chdir will help switch between Training and Testing Files for the next few sections
os.chdir('/content/Testing')

In [ ]:
# Good for making sure the directory has changed
!ls

# **Counting Images Within Each Class**

This will help to see if there is any imbalances in the dataset since an imbalanced data can lead to bias and low accuracy models.

In [ ]:
# function that counts images in each class
def count_files(dir_path):
  count = 0
  for path in os.listdir(dir_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dir_path, path)):
        count += 1
  return count





In [ ]:
# counting the number of images in each class in the Training Set

print("This is the testing set")

dir_path = r"meningioma"
meningioma_count = count_files(dir_path)
print("meningioma: ",meningioma_count)

dir_path = r"glioma"
glioma_count = count_files(dir_path)
print("glioma: ", glioma_count)

dir_path = r"notumor"
notumor_count = count_files(dir_path)
print("notumor: ", notumor_count)

dir_path = r"pituitary"
pituitary_count = count_files(dir_path)
print("pituitary: ",pituitary_count)

In [ ]:
# Creating dataframe for our classes in testing data

d = {'class': ['meningioma', 'glioma', 'notumor' , 'pituitary' ], 'count' : [meningioma_count, 
                                                                              glioma_count,
                                                                              notumor_count, 
                                                                              pituitary_count]}
df = pd.DataFrame(data=d)

In [ ]:
df.head()

In [ ]:
os.chdir('/content/Training')

In [ ]:
# counting the number of images in each class in the Training Set

print("This is the training set")

dir_path = r"meningioma"
meningioma_count = count_files(dir_path)
print("meningioma: ", meningioma_count)

dir_path = r"glioma"
glioma_count = count_files(dir_path)
print("glioma: ", glioma_count)

dir_path = r"notumor"
notumor_count = count_files(dir_path)
print("notumor: ", notumor_count)

dir_path = r"pituitary"
pituitary_count = count_files(dir_path)
print("pituitary: ", pituitary_count)

In [ ]:
# Creating dataframe for classes in training data
d = {'class': ['meningioma', 'glioma', 'notumor' , 'pituitary' ], 'count' : [meningioma_count, 
                                                                              glioma_count,
                                                                              notumor_count, 
                                                                              pituitary_count]}
df = pd.DataFrame(data=d)

In [ ]:
df.head()

# **Plotting the Number of Images in each Class for Testing and Training Data**

In [ ]:
# Using seaborn to create a barplot to plot the number of images within each of the classes in Testing Set
sns.barplot(x="class", y='count', data=df).set(title="Brain Tumor Classes in Testing Set");

In [ ]:
# # Using seaborn to create a barplot to plot the number of images within each of the classes in Training set
sns.barplot(x="class", y='count', data=df).set(title="Brain Tumor Classes in Training Set");

After looking at both the testing and training counts, notumor has quite a few more images than the other classes which could potentially lead to bias. Other than this, the other classes are well balanced which will help with having high accuracy models.

# **Displaying Testing Images🖼**

Now it's time to see what the images in the dataset looks like. This will help with understanding how each image will be classified into their respective class.

In [ ]:
# Displaying glioma image

glioma_img = "/content/Testing/glioma/Te-gl_0011.jpg"
print("Glioma Testing Image: ")
glioma_img = cv2.imread(glioma_img)
plt.imshow(glioma_img);

In [ ]:
# Displaying meningioma image

meningioma_img = "/content/Testing/meningioma/Te-meTr_0007.jpg"
print("Meningioma Testing Image: ")
meningioma_img = cv2.imread(meningioma_img)

In [ ]:
# displaying notumor image

notumor_img = "/content/Testing/notumor/Te-noTr_0003.jpg"
print("notumor Testing Image: ")
notumor_img = cv2.imread(notumor_img)
plt.imshow(notumor_img);

In [ ]:
# displaying pituitary image

pituitary_img = "/content/Testing/pituitary/Te-piTr_0004.jpg"
print("Pituitary Testing Image: ")
pituitary_img = cv2.imread(pituitary_img) 
plt.imshow(pituitary_img);

# **Displaying Training Images🖼**

In [ ]:
os.chdir('/content/Training')

In [ ]:
img = "meningioma/Tr-me_0560.jpg"
img = cv2.imread(img)
print("Meningioma Training Image: ")
plt.imshow(img);

In [ ]:
img = "/content/Training/glioma/Tr-glTr_0005.jpg"
img = cv2.imread(img)
print("Glioma Training Image: ")
plt.imshow(img);

In [ ]:
img = "/content/Training/notumor/Tr-noTr_0008.jpg"
img = cv2.imread(img) # reads image
print("notumor Training Image: ")
plt.imshow(img);

In [ ]:
img = "/content/Training/pituitary/Tr-piTr_0009.jpg"
img = cv2.imread(img)
print("Pituitary Training Image: ")
plt.imshow(img);

In [ ]:
print("Image Dimensions:",img.shape)

In [ ]:
img = "/content/Training/notumor/Tr-noTr_0004.jpg"
img = cv2.imread(img) # reads image
print("notumor Training Image: ")
plt.imshow(img);

After displaying these images, some of the features that can help distinguish the differences are the location, shape, and size since Pituitary tumors tend to be much smaller than others.

# **Image Augmentation**

Image Augmentation can be helpful because it extracts the objects of our interest for further processing such as description or recognition. It also gives more ways to explore the images in the dataset with different options like zooming in, resizing, gray scaling, and blurring.

In [ ]:
# resizing to see the tumor better
small_img = cv2.resize(img,(455,256))
big_img = cv2.resize(img,(910,511))
cv2_imshow(big_img)

In [ ]:
# zooming in

zoom = 0.9

centerX,centerY=int(img.shape[0]*0.75),int(img.shape[1]/2)
radiusX,radiusY= int((1-zoom)*img.shape[0]*2),int((1-zoom)*img.shape[1]*2)

minX,maxX=centerX-radiusX,centerX+radiusX
minY,maxY=centerY-radiusY,centerY+radiusY

cropped = img[minX:maxX, minY:maxY]
zoom_img = cv2.resize(cropped, (img.shape[1], img.shape[0])) 
cv2_imshow(zoom_img)

In [ ]:
# converting to gray scale

gray_image = color.rgb2gray(img)


In [ ]:
plt.imshow(gray_image);

In [ ]:
print("Original Image Dimensions: ",img.shape)
print("Gray Image Dimensions: ",gray_image.shape)

In [ ]:
# blurring image

blur_img = cv2.blur(img, (5,5))
cv2_imshow(blur_img)


In [ ]:
# flipping image
img = cv2.imread('/content/Training/pituitary/Tr-piTr_0009.jpg')
img = cv2.flip(img, 0)
cv2_imshow(img)

# **Preparing Data for Model Development**

In [ ]:
# Storing file paths into variables
train_path = '/content/Training'
test_path = '/content/Testing'

In [ ]:
X = []
y = []
image_size = 150
for i in labels:
  folderpath = os.path.join(train_path, i)
  for j in tqdm(os.listdir(folderpath)):
    img = cv2.imread(os.path.join(folderpath, j))
    img = cv2.resize(img, (image_size, image_size))
    X.append(img)
    y.append(labels[i])

for i in labels:
  folderpath = os.path.join(test_path, i)
  for j in tqdm(os.listdir(folderpath)):
    img = cv2.imread(os.path.join(folderpath, j))
    img = cv2.resize(img, (image_size, image_size))
    X.append(img)
    y.append(labels[i])

X = np.array(X)
y = np.array(y)

Before this section, we looked at the shape of an image, in which the dimensions were (512,512,3). We changed the image size to 150 and now the new image size is (150,150,3). 

This is because it is faster to train models due to the reduced number of parameters.

In [ ]:
X = X.reshape(7023, 3*150*150)

In [ ]:
dir_path = X

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
df = pd.DataFrame(np.c_[X,y])

In [ ]:
df.head()

In [ ]:
df[67500].value_counts()

In [ ]:
sns.countplot(x=df[67500]);

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("X_train shape", X_train.shape)
print("X test shape",X_test.shape)
print("y train shape",y_train.shape)
print("y test shape",y_test)

# **Building Models**
**The Classification Models that will be used are:**
- Logistic Regression
- Ridge Classifier
- Random Forest Classifier
- Decision Tree Classifier
- Support Vector Classifier
- K Neighbors Classifier
- Convolutional Neural Network
- Transfer Learning (AlexNet and MobileNet)
- Cross Validation Convolutional Neural Network

**After building these models, some of the different metrics that will be used to evaluate model performance will be:**
- Accuracy Score - (Number of Correct Predictions/Total Predictions)

- F1 score - 2*(precision**recall)/(precision+recall)

- Recall - True Positives / (True Positives + False Negatives)

- Precision - True Positives / (True Positives + False Positives)

- ROC/AUC score - False Positive Rate = False Positives/(False Positives+True Negatives), True Positive Rate = True Positives/(True Positives+False Negatives)

- Cohen Kappa Score - (Total accuracy – Random accuracy) / (1- Random accuracy).

True Positive - Prediction is no tumor, and there is no tumor.

True Negative - Prediction is a tumor, and there is a tumor.

False Positive - Prediction is tumor, but there is no tumor.

False Negative - Prediction is no tumor but there is a tumor.

These metrics all have a different way of evaluating the models predictions based on different factors such as data imbalance, the type of predictions being made, and the models ability to differentiate between each class. The goal is to aim for no false negative predictions.

In [ ]:
# Logistic Regression

lr_model = LogisticRegression()

lr_model.fit(X_train, y_train)

preds = lr_model.predict(X_test)

print("Logistic Regression Accuracy:",accuracy_score(y_test, preds))
print("Logistic Regression f1 score:",f1_score(y_test, preds, average='weighted'))
print("Logistic Regression Recall Score:",recall_score(y_test, preds, average='weighted'))
print("Logistic Regression Precision Score:",precision_score(y_test, preds, average='weighted'))
print("Logistic Regression Kappa score",cohen_kappa_score(y_test, preds))
# Accuracy:0.86
# f1 score: 0.86
# recall:0.86
# precision:0.86
# kappa score:0.81
#took 3m 14s to finish running

In [ ]:
# Ridge Classifier

rc_model = RidgeClassifier()

rc_model.fit(X_train, y_train)

preds = rc_model.predict(X_test)

print("Ridge Classifier Accuracy:",accuracy_score(y_test, preds))
print("Ridge Classifier f1 score:",f1_score(y_test, preds, average='weighted'))
print("Ridge Classifier Recall Score:",recall_score(y_test, preds, average='weighted'))
print("Ridge Classifier Precision Score:",precision_score(y_test, preds, average='weighted'))
print("Ridge Classifier Kappa score",cohen_kappa_score(y_test, preds))

In [ ]:
# Random Forest Classifier

rf_model = RandomForestClassifier(max_depth=5, random_state=0)

rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)

print("RandomForest Accuracy:",accuracy_score(y_test, preds))
print("RandomForest f1 score:",f1_score(y_test, preds, average='weighted'))
print("RandomForest Recall Score:",recall_score(y_test, preds, average='weighted'))
print("RandomForest Precision Score:",precision_score(y_test, preds, average='weighted'))
print("RandomForest Kappa score",cohen_kappa_score(y_test, preds))

In [ ]:
# Decision Tree Classifier

dt_model = DecisionTreeClassifier(random_state=0)

dt_model.fit(X_train, y_train)

preds = dt_model.predict(X_test)

print("Decision Tree Accuracy:",accuracy_score(y_test, preds))
print("Decision Tree f1 score:",f1_score(y_test, preds, average='weighted'))
print("Decision Tree Recall Score:",recall_score(y_test, preds, average='weighted'))
print("Decision Tree Precision Score:",precision_score(y_test, preds, average='weighted'))
print("Decision Tree Kappa score",cohen_kappa_score(y_test, preds))

In [ ]:
# Support Vector Classifier

svc_model = svm.SVC()

svc_model.fit(X_train, y_train)

preds = svc_model.predict(X_test)

print("Support Vector Classifier Accuracy:",accuracy_score(y_test, preds))
print("Support Vector Classifier f1 score:",f1_score(y_test, preds, average='micro'))
print("Support Vector Classifier Recall Score:",recall_score(y_test, preds, average='micro'))
print("Support Vector Classifier Precision Score:",precision_score(y_test, preds, average='micro'))
print("Support Vector Classifier Kappa score",cohen_kappa_score(y_test, preds))

In [ ]:
# K Neighbors Classifier

knn_model = KNeighborsClassifier(n_neighbors=3)

knn_model.fit(X_train, y_train)

preds = knn_model.predict(X_test)

print("KNeighborsClassifier Accuracy:",accuracy_score(y_test, preds))
print("KNeighborsClassifier f1 score:",f1_score(y_test, preds, average='weighted'))
print("KNeighborsClassifier Recall Score:",recall_score(y_test, preds, average='weighted'))
print("KNeighborsClassifier Precision Score:",precision_score(y_test, preds, average='weighted'))
print("KNeighborsCLassifier Kappa score",cohen_kappa_score(y_test, preds))

In [ ]:
X_train_reshaped = X_train.reshape(5618, 150,150,3)
X_test_reshaped = X_test.reshape(1405,150,150,3)
print("New X train shape:",X_train_reshaped.shape)
print("New X test shape:",X_test_reshaped.shape)

In [ ]:
# Convolutional Neural Network

cnn_model = Sequential()

cnn_model.add(Conv2D(32,(3,3), padding='same', input_shape=(image_size, image_size,3)))
cnn_model.add(Activation('relu'))
cnn_model.add(Conv2D(32, (3,3), padding='same'))
cnn_model.add(Activation('relu'))
cnn_model.add(MaxPooling2D(pool_size=(2,2)))
cnn_model.add(Conv2D(32, (3,3), padding='same'))
cnn_model.add(Activation('relu'))
cnn_model.add(Conv2D(32, (3,3), padding='same'))
cnn_model.add(Activation('relu'))
cnn_model.add(Conv2D(64, (3,3), padding='same'))
cnn_model.add(Activation('relu'))
cnn_model.add(MaxPooling2D(pool_size=(2,2)))
cnn_model.add(Conv2D(64, (3,3), padding='same'))
cnn_model.add(Activation('relu'))
cnn_model.add(Dropout(0.2))
cnn_model.add(Dense(512))
cnn_model.add(Activation('relu'))
cnn_model.add(MaxPooling2D(pool_size=(2,2)))
cnn_model.add(Dropout(0.2))
cnn_model.add(Flatten())
cnn_model.add(Dense(4))
cnn_model.add(Activation('softmax'))

opt = keras.optimizers.RMSprop(learning_rate=0.001, decay=1e-5)

cnn_model.compile(loss='sparse_categorical_crossentropy', 
                  optimizer='adam',
                  metrics=['accuracy'])

history = cnn_model.fit(X_train_reshaped, y_train,
                        validation_data=(X_test_reshaped, y_test),
                        epochs=30)

cnn_model.summary()

Next, we will be trying out MobileNet and AlexNet, which are transfer learning models. They are different than a CNN and the other models because we use a pre-trained model as the starting point for a model on a new task. 

Look below to see what the architecture of AlexNet and MobileNet look like.

<!-- Image removed: run notebook to regenerate -->

In [ ]:
# Mobilenet

mn_model = MobileNet(input_shape=(image_size,image_size,3), include_top=False, pooling='max')
tl_model = Sequential()
tl_model.add(mn_model)
tl_model.add(Dropout(0.2))
tl_model.add(Dense(512, activation='relu'))
tl_model.add(Dropout(0.2))
tl_model.add(Dense(4, activation="softmax"))

opt = keras.optimizers.RMSprop(learning_rate=0.001, decay=1e-5)


tl_model.compile(loss='sparse_categorical_crossentropy', 
                  optimizer='adam',
                  metrics=['accuracy'])

history = tl_model.fit(X_train_reshaped, y_train,
                        validation_data=(X_test_reshaped, y_test),
                        epochs=30)

tl_model.summary()

In [ ]:
y_pred = tl_model.predict(X_test_reshaped)
print("MobileNet Recall:",recall_score(y_test.tolist(), y_pred.argmax(axis=1).tolist(), average='weighted'))
print("MobileNet Precision:",precision_score(y_test.tolist(), y_pred.argmax(axis=1).tolist(), average='weighted'))
print("MobileNet f1 score:",f1_score(y_test.tolist(), y_pred.argmax(axis=1).tolist(), average='weighted'))
print("MobileNet Kappa Score:",cohen_kappa_score(y_test.tolist(), y_pred.argmax(axis=1).tolist()))

In [ ]:
# AlexNet

an_model = Sequential()

an_model.add(Conv2D(96, 11,strides = 4))
an_model.add(Activation('relu'))
an_model.add(Conv2D(256, 5))
an_model.add(Activation('relu'))
an_model.add(MaxPooling2D(2))
an_model.add(Conv2D(384, 3, padding='same'))
an_model.add(Activation('relu'))
an_model.add(Conv2D(256, 3, padding='same'))
an_model.add(Activation('relu'))
an_model.add(MaxPooling2D(2))
an_model.add(Activation('relu'))
an_model.add(Dropout(0.5))
an_model.add(Dense(4096))
an_model.add(Activation('relu'))
an_model.add(Dropout(0.3))
an_model.add(Dense(128))
an_model.add(Activation('relu'))
an_model.add(Flatten())
an_model.add(Dense(4))
an_model.add(Activation('softmax'))

opt = keras.optimizers.RMSprop(learning_rate=0.001,decay=1e-5)

an_model.compile(loss='sparse_categorical_crossentropy',
                 optimizer='adam',
                 metrics=['accuracy'])

history = an_model.fit(X_train_reshaped.astype(np.float32), y_train.astype(np.float32), 
                    validation_data=(X_test_reshaped.astype(np.float32), y_test.astype(np.float32)),
                    epochs=30)

an_model.summary()

# **Cross Validation**

Cross-validation is a resampling method that uses different portions of the data to test and train a model on different iterations.

In this case, the same CNN model from before will be used but the inputs will be the new cross validation variables.

In [ ]:
# https://www.kaggle.com/code/franklemuchahary/basic-cnn-keras-with-cross-validation

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_train_reshaped, y_train, test_size=0.33, random_state=101)

In [ ]:
# Cross Val CNN

cv_cnn = Sequential()

cv_cnn.add(Conv2D(32,(3,3), padding='same', input_shape=(image_size, image_size,3)))
cv_cnn.add(Activation('relu'))
cv_cnn.add(Conv2D(32, (3,3), padding='same'))
cv_cnn.add(Activation('relu'))
cv_cnn.add(MaxPooling2D(pool_size=(2,2)))
cv_cnn.add(Conv2D(32, (3,3), padding='same'))
cv_cnn.add(Activation('relu'))
cv_cnn.add(Conv2D(32, (3,3), padding='same'))
cv_cnn.add(Activation('relu'))
cv_cnn.add(Conv2D(64, (3,3), padding='same'))
cv_cnn.add(Activation('relu'))
cv_cnn.add(MaxPooling2D(pool_size=(2,2)))
cv_cnn.add(Conv2D(64, (3,3), padding='same'))
cv_cnn.add(Activation('relu'))
cv_cnn.add(Dropout(0.2))
cv_cnn.add(Dense(512))
cv_cnn.add(Activation('relu'))
cv_cnn.add(MaxPooling2D(pool_size=(2,2)))
cv_cnn.add(Dropout(0.2))
cv_cnn.add(Flatten())
cv_cnn.add(Dense(4))
cv_cnn.add(Activation('softmax'))

opt = keras.optimizers.RMSprop(learning_rate=0.001, decay=1e-5)

cv_cnn.compile(loss='sparse_categorical_crossentropy', 
                  optimizer='adam',
                  metrics=['accuracy'])

history = cv_cnn.fit(X_train_v, y_train_v,
                        validation_data=(X_test_v, y_test_v),
                        epochs=30)

cv_cnn.summary()

Out of all these models, the MobileNet model performed best, with a validation accuracy **98.01**. This is because it significantly reduces the number of parameters when compared to the network with regular convolutions with the same depth in the nets and it is a pre trained model.

# **Using Confusion Matrices to Evaluate the Models Further**

Confusion Matrices are used to determine classification models, and gives a better understanding of what the model is getting right and what errors it is making.

The different types of predictions can be either **True Positive, True Negative, False Positive, or False Negative**. What we hope to see, is no false negatives.

In [ ]:
# KNN CM
cm = plot_confusion_matrix(knn_model, X_test, y_test);
plt.title("KNN Model Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show(cm)

In [ ]:
# RidgeClassifier CM
cm = plot_confusion_matrix(rc_model, X_test, y_test)
plt.title("RidgeClassifier Model Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show(cm)

In [ ]:
# Logistic Regression CM
cm = plot_confusion_matrix(lr_model, X_test, y_test)
plt.title("Logistic Regression Model Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show(cm)

In [ ]:
# Random Forest CM
cm = plot_confusion_matrix(rf_model, X_test, y_test)
plt.title("Random Forest Model Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show(cm)

In [ ]:
# Decision Tree CM
cm = plot_confusion_matrix(dt_model, X_test, y_test)
plt.title("Decision Tree Model Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show(cm)

In [ ]:
# CNN CM

y_pred = cnn_model.predict(X_test_reshaped)
cm = confusion_matrix(y_test.tolist(), y_pred.argmax(axis=1).tolist())
disp = ConfusionMatrixDisplay(cm);
disp.plot()
plt.title("CNN Confusion Matrix")
plt.show()

In [ ]:
# MobileNet CM
y_pred = tl_model.predict(X_test_reshaped)
cm = confusion_matrix(y_test.tolist(), y_pred.argmax(axis=1).tolist())
disp = disp = ConfusionMatrixDisplay(cm);
disp.plot()
plt.title("MobileNet Confusion Matrix")
plt.show()

In [ ]:
# AlexNet CM
y_pred = an_model.predict(X_test_reshaped)
cm = confusion_matrix(y_test.tolist(), y_pred.argmax(axis=1).tolist())
disp = ConfusionMatrixDisplay(cm);
disp.plot()
plt.title("AlexNet Confusion Matrix")
plt.show()

# **ROC/AUC Scores**

The Area Under the Curve (AUC) is the measure of the ability of a classifier to distinguish between classes and is used as a summary of the ROC curve. The higher the AUC, the better the performance of the model at distinguishing between the positive and negative classes.

In [ ]:
# Logistic Regression
y_preb_probs = lr_model.predict_proba(X_test)
print("Logistic Regression ROC/AUC score:",roc_auc_score(y_test, y_preb_probs, average="weighted", multi_class="ovr"))

In [ ]:
# Random Forest
y_preb_probs = rf_model.predict_proba(X_test)
print("RandomForest ROC/AUC score:",roc_auc_score(y_test, y_preb_probs, average="weighted", multi_class="ovr"))

In [ ]:
# Decision Tree
y_preb_probs = dt_model.predict_proba(X_test)
print("Decision Tree ROC/AUC score:",roc_auc_score(y_test, y_preb_probs, average="weighted", multi_class="ovr"))

In [ ]:
# CNN
y_preb_probs = cnn_model.predict(X_test_reshaped)
print("CNN ROC/AUC score:",roc_auc_score(y_test, y_preb_probs, average="weighted", multi_class="ovr"))

In [ ]:
# Support Vector Classifier
y_preb_probs = svc_model.predict_proba(X_test)
print("Support Vector Classifier ROC/AUC score:",roc_auc_score(y_test, y_preb_probs, average="weighted", multi_class="ovr"))

In [ ]:
# KNN 
y_preb_probs = knn_model.predict_proba(X_test)
print("KNN ROC/AUC score:",roc_auc_score(y_test, y_preb_probs, average="weighted", multi_class="ovr"))

In [ ]:
# MobileNet

y_preb_probs = tl_model.predict(X_test_reshaped)
print("MobileNet ROC/AUC Score:",roc_auc_score(y_test, y_preb_probs, average='weighted', multi_class="ovr"))

In [ ]:
# AlexNet

y_preb_probs = an_model.predict(X_test_reshaped)
print("AlexNet ROC/AUC Score:",roc_auc_score(y_test, y_preb_probs, average='weighted', multi_class="ovr"))

In [ ]:
# Cross Validation CNN 
y_preb_probs = cv_cnn.predict(X_test_v)
print("Cross Val CNN ROC/AUC Score:",roc_auc_score(y_test_v, y_preb_probs, average='weighted', multi_class="ovr"))

# **Building a Web App📲**

It's now time to deploy the MobileNet model to a web app using Streamlit. Streamlit is a fast way to create web apps for maching learning models with Python.

We'll also be using Ngrok, which will allow us to actually access the website that we have created.

The webapp will allow users to upload an image of a Brain MRI and receive a prediction classifiying what the tumor is.

Some of the problems that we could see were there were some classes that were predicted wrong such as meningioma and glioma which could be something that we could see if a glioma or meningioma image is uploaded.


In [ ]:
# save model
dump(tl_model, "model.joblib")

# **End of Notebook📙**

This dataset is sourced from: Msoud Nickparvar. (2021). <i>Brain Tumor MRI Dataset</i> [Data set]. Kaggle. https://doi.org/10.34740/KAGGLE/DSV/2645886